In [ ]:
from pathlib import Path

raw_folder = Path.home() / "Desktop" / "Person2_Work"
registry_folder = Path.home() / "Downloads" / "BMI5207_Person4_PRIVATE" / "private"

excel_files = [
    p for p in raw_folder.glob("*.xlsx")
    if not p.name.startswith("~$")
]

print("Excel files found:", len(excel_files))

for name in [
    "person_registry.csv",
    "visit_registry.csv",
    "source_manifest.csv",
    "unlinked_source_cases.csv",
]:
    print(name, "OK" if (registry_folder / name).is_file() else "NOT FOUND")

In [ ]:
import openpyxl

total_rows = 0

for number, file in enumerate(sorted(excel_files), start=1):
    workbook = openpyxl.load_workbook(
        file, read_only=True, data_only=True
    )

    for sheet in workbook:
        rows = [
            row for row in sheet.iter_rows(values_only=True)
            if any(value is not None for value in row)
        ]

        record_count = max(len(rows) - 1, 0)
        column_count = len(rows[0]) if rows else 0
        total_rows += record_count

        print(
            f"File {number} | Sheet: {sheet.title} | "
            f"Records: {record_count} | Columns: {column_count}"
        )

    workbook.close()

print("Total source records:", total_rows)

In [ ]:
raw_records = []
expected_headers = None

for file in sorted(excel_files):
    workbook = openpyxl.load_workbook(
        file, read_only=True, data_only=True
    )

    for sheet in workbook:
        rows = [
            row for row in sheet.iter_rows(values_only=True)
            if any(value is not None for value in row)
        ]

        if not rows:
            continue

        headers = tuple(rows[0])

        if expected_headers is None:
            expected_headers = headers
        elif headers != expected_headers:
            raise ValueError("Column names or order do not match.")

        for row in rows[1:]:
            raw_records.append(dict(zip(headers, row)))

    workbook.close()

unique_records = []
seen = set()

for record in raw_records:
    fingerprint = tuple(record[name] for name in expected_headers)

    if fingerprint not in seen:
        seen.add(fingerprint)
        unique_records.append(record)

print("Original records:", len(raw_records))
print("Exact duplicates removed:", len(raw_records) - len(unique_records))
print("Cleaned records:", len(unique_records))

In [ ]:
import csv

def read_csv_file(path):
    with path.open(encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))

person_registry = read_csv_file(
    registry_folder / "person_registry.csv"
)
visit_registry = read_csv_file(
    registry_folder / "visit_registry.csv"
)

person_lookup = {
    row["Patient_No"]: row
    for row in person_registry
}

visit_lookup = {
    (row["Patient_No"], row["Case_No"]): row
    for row in visit_registry
}

assert len(person_lookup) == len(person_registry), "Duplicate patient keys."
assert len(visit_lookup) == len(visit_registry), "Duplicate visit keys."

unmatched_records = []
inconsistent_ids = []

for record in unique_records:
    patient = str(record["Patient_No"])
    case = str(record["Case_No"])

    person = person_lookup.get(patient)
    visit = visit_lookup.get((patient, case))

    if person is None or visit is None:
        unmatched_records.append((patient, case))
    elif person["person_id"] != visit["person_id"]:
        inconsistent_ids.append((patient, case))

print("Patients in registry:", len(person_lookup))
print("Visits in registry:", len(visit_lookup))
print("Unmatched records:", len(unmatched_records))
print("Inconsistent person IDs:", len(inconsistent_ids))

assert not unmatched_records, "Some source records could not be matched."
assert not inconsistent_ids, "Person IDs disagree between registries."

print("Registry linkage check passed.")

In [ ]:
from datetime import datetime, timedelta

cleaned_visits = []

date_fields = [
    "Adm_DateTime",
    "Dis_DateTime",
    "Dis_DateTime_Planned",
]

for record in unique_records:
    patient = str(record["Patient_No"])
    case = str(record["Case_No"])

    person = person_lookup[patient]
    visit = visit_lookup[(patient, case)]
    offset = timedelta(days=int(person["date_offset_days"]))

    # Create a new record without the original patient/case identifiers.
    cleaned = {
        name: value
        for name, value in record.items()
        if name not in ("Patient_No", "Case_No")
    }

    cleaned["person_id"] = int(person["person_id"])
    cleaned["visit_occurrence_id"] = int(visit["visit_occurrence_id"])

    for field in date_fields:
        original_date = record[field]

        if original_date is None:
            cleaned[field] = None
        else:
            if not isinstance(original_date, datetime):
                raise ValueError(f"Unexpected date format in {field}.")

            cleaned[field] = original_date + offset

    original_duration = record["Dis_DateTime"] - record["Adm_DateTime"]
    shifted_duration = cleaned["Dis_DateTime"] - cleaned["Adm_DateTime"]

    assert original_duration == shifted_duration, "Visit duration changed."
    assert shifted_duration >= timedelta(0), "Discharge precedes admission."

    cleaned_visits.append(cleaned)

assert len({row["visit_occurrence_id"] for row in cleaned_visits}) == 21
assert all(
    "Patient_No" not in row and "Case_No" not in row
    for row in cleaned_visits
)

print("Cleaned visits:", len(cleaned_visits))
print("Unique patients:", len({row["person_id"] for row in cleaned_visits}))
print("Shared IDs applied.")
print("Dates shifted; visit durations preserved.")
print("Original patient/case identifiers excluded.")

In [ ]:
import csv
from datetime import datetime

output_folder = raw_folder / "outputs"
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "cleaned_case_visits.csv"

def csv_value(value):
    if isinstance(value, datetime):
        return value.isoformat(sep=" ", timespec="seconds")
    return value

with output_file.open(
    "w", encoding="utf-8-sig", newline=""
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(cleaned_visits[0].keys())
    )
    writer.writeheader()

    for record in cleaned_visits:
        writer.writerow({
            name: csv_value(value)
            for name, value in record.items()
        })

# Read the saved file back to verify it.
with output_file.open(
    encoding="utf-8-sig", newline=""
) as file:
    saved_records = list(csv.DictReader(file))

assert len(saved_records) == 21
assert "Patient_No" not in saved_records[0]
assert "Case_No" not in saved_records[0]

print("Saved records:", len(saved_records))
print("File size:", output_file.stat().st_size, "bytes")
print("Saved to:", output_file.resolve())

In [ ]:
visit_occurrence = []

for record in cleaned_visits:
    start = record["Adm_DateTime"]
    end = record["Dis_DateTime"]
    source_type = record["Adm_Type_Desc"]

    if source_type == "Emergency" and end.date() > start.date():
        visit_concept_id = 9201  # Inpatient Visit

    elif (
        source_type == "Day Surgery"
        and end.date() == start.date()
        and record["DS_Turn_Inpatient_Ind"] is False
    ):
        visit_concept_id = 9202  # Outpatient Visit

    else:
        raise ValueError("Visit classification requires manual review.")

    visit_occurrence.append({
        "visit_occurrence_id": record["visit_occurrence_id"],
        "person_id": record["person_id"],
        "visit_concept_id": visit_concept_id,
        "visit_start_date": start.date().isoformat(),
        "visit_start_datetime": csv_value(start),
        "visit_end_date": end.date().isoformat(),
        "visit_end_datetime": csv_value(end),
        "visit_type_concept_id": 32817,
        "provider_id": None,
        "care_site_id": None,
        "visit_source_value": source_type,
        "visit_source_concept_id": 0,
        "admitted_from_concept_id": 0,
        "admitted_from_source_value": record["Referral_Type_From"],
        "discharged_to_concept_id": 0,
        "discharged_to_source_value": record["Discharge_Type_Desc"],
        "preceding_visit_occurrence_id": None,
    })

# Link each visit to the patient's preceding visit.
visit_occurrence.sort(
    key=lambda row: (
        row["person_id"],
        row["visit_start_datetime"],
        row["visit_occurrence_id"],
    )
)

previous_visit = {}

for row in visit_occurrence:
    person_id = row["person_id"]
    row["preceding_visit_occurrence_id"] = previous_visit.get(person_id)
    previous_visit[person_id] = row["visit_occurrence_id"]

visit_file = output_folder / "visit_occurrence_provisional.csv"

with visit_file.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(visit_occurrence[0].keys())
    )
    writer.writeheader()
    writer.writerows(visit_occurrence)

with visit_file.open(encoding="utf-8-sig", newline="") as file:
    saved_visits = list(csv.DictReader(file))

assert len(saved_visits) == 21
assert len({row["visit_occurrence_id"] for row in saved_visits}) == 21

print("Saved visits:", len(saved_visits))
print("Inpatient visits:", sum(
    row["visit_concept_id"] == "9201" for row in saved_visits
))
print("Outpatient visits:", sum(
    row["visit_concept_id"] == "9202" for row in saved_visits
))
print("Saved to:", visit_file.resolve())

In [ ]:
diagnosis_events = []
duplicate_diagnoses = []
token_count = 0

for record in cleaned_visits:
    primary = str(
        record["Primary_Diagnosis_Code_Mediclaim"] or ""
    ).strip().upper()

    secondary = [
        code.strip().upper()
        for code in str(
            record["Secondary_Diagnosis_Codes_Subvention"] or ""
        ).split(",")
        if code.strip()
    ]

    assert len(secondary) == int(
        record["No_of_Secondary_Diagnosis_Subvention"]
    ), "Secondary diagnosis count does not match."

    tokens = []

    if primary:
        tokens.append((
            primary,
            "Primary",
            record["Primary_Diagnosis_Description_Mediclaim"],
        ))

    tokens.extend(
        (code, "Secondary", "")
        for code in secondary
    )

    seen_codes = set()

    for position, (code, role, description) in enumerate(tokens, start=1):
        token_count += 1

        if code in seen_codes:
            duplicate_diagnoses.append({
                "visit_occurrence_id": record["visit_occurrence_id"],
                "source_code": code,
                "diagnosis_role": role,
                "source_position": position,
                "reason": "Repeated source code within the same visit",
            })
            continue

        seen_codes.add(code)

        diagnosis_events.append({
            "diagnosis_event_id": len(diagnosis_events) + 1,
            "person_id": record["person_id"],
            "visit_occurrence_id": record["visit_occurrence_id"],
            "diagnosis_role": role,
            "source_code": code,
            "source_description": description,
            "source_position": position,
            "date_proxy": record["Adm_DateTime"].date().isoformat(),
            "date_basis": "Shifted admission date; not confirmed onset",
        })

assert token_count == 149
assert len(duplicate_diagnoses) == 2
assert len(diagnosis_events) == 147

diagnosis_file = output_folder / "diagnosis_events.csv"

with diagnosis_file.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(diagnosis_events[0].keys())
    )
    writer.writeheader()
    writer.writerows(diagnosis_events)

with diagnosis_file.open(encoding="utf-8-sig", newline="") as file:
    saved_diagnoses = list(csv.DictReader(file))

assert len(saved_diagnoses) == 147

print("Original diagnosis tokens:", token_count)
print("Duplicate tokens removed:", len(duplicate_diagnoses))
print("Saved diagnosis events:", len(saved_diagnoses))
print("Distinct source codes:", len({
    row["source_code"] for row in diagnosis_events
}))
print("Saved to:", diagnosis_file.resolve())

In [ ]:
external_codes = set(
    "U739 W010 X59 Y408 Y409 Y442 Y531 Y9209 Y9222 Y929".split()
)

context_codes = set(
    "Z312 Z599 Z720 Z8643 Z921 Z952 Z954".split()
)

mapping_inventory = {}

for event in diagnosis_events:
    code = event["source_code"]

    if code not in mapping_inventory:
        if code == "M85503":
            category = "Morphology review"
        elif code in external_codes:
            category = "External cause or context review"
        elif code in context_codes:
            category = "History, status or service review"
        else:
            category = "Condition candidate"

        # Query formatting only; the source coding system is unconfirmed.
        query_code = (
            "8550/3" if code == "M85503"
            else code[:3] + "." + code[3:] if len(code) > 3
            else code
        )

        mapping_inventory[code] = {
            "source_code": code,
            "query_candidate": query_code,
            "event_count": 0,
            "primary_count": 0,
            "secondary_count": 0,
            "source_description": "",
            "review_category": category,
            "confirmed_source_vocabulary": "",
            "target_concept_id": "",
            "target_concept_name": "",
            "target_domain": "",
            "mapping_status": "Pending review",
        }

    item = mapping_inventory[code]
    item["event_count"] += 1

    if event["diagnosis_role"] == "Primary":
        item["primary_count"] += 1
    else:
        item["secondary_count"] += 1

    if event["source_description"]:
        item["source_description"] = event["source_description"]

mapping_rows = [
    mapping_inventory[code]
    for code in sorted(mapping_inventory)
]

assert len(mapping_rows) == 72
assert sum(row["event_count"] for row in mapping_rows) == 147

mapping_file = output_folder / "diagnosis_mapping_worklist.csv"

with mapping_file.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(mapping_rows[0].keys())
    )
    writer.writeheader()
    writer.writerows(mapping_rows)

with mapping_file.open(encoding="utf-8-sig", newline="") as file:
    saved_mapping = list(csv.DictReader(file))

assert len(saved_mapping) == 72

print("Source codes in worklist:", len(saved_mapping))
print("Diagnosis events covered:", sum(
    int(row["event_count"]) for row in saved_mapping
))
print("Saved to:", mapping_file.resolve())

In [ ]:
from datetime import date

item = mapping_inventory["J189"]

item["target_concept_id"] = 255848
item["target_concept_name"] = "Pneumonia"
item["target_domain"] = "Condition"
item["mapping_status"] = (
    "Athena mapping verified; local source vocabulary pending"
)

# Keep local source vocabulary unconfirmed.
item["confirmed_source_vocabulary"] = ""

# Add evidence columns consistently to all 72 records.
for row in mapping_inventory.values():
    row.setdefault("evidence_source_vocabulary", "")
    row.setdefault("evidence_source_concept_id", "")
    row.setdefault("target_vocabulary", "")
    row.setdefault("target_standard_status", "")
    row.setdefault("target_validity", "")
    row.setdefault("source_evidence_url", "")
    row.setdefault("target_evidence_url", "")
    row.setdefault("checked_date", "")

item.update({
    "evidence_source_vocabulary": "ICD10",
    "evidence_source_concept_id": 45601123,
    "target_vocabulary": "SNOMED",
    "target_standard_status": "Standard",
    "target_validity": "Valid",
    "source_evidence_url":
        "https://athena.ohdsi.org/search-terms/terms/45601123",
    "target_evidence_url":
        "https://athena.ohdsi.org/search-terms/terms/255848",
    "checked_date": date.today().isoformat(),
})

mapping_rows = [
    mapping_inventory[code]
    for code in sorted(mapping_inventory)
]

with mapping_file.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(
        file, fieldnames=list(mapping_rows[0].keys())
    )
    writer.writeheader()
    writer.writerows(mapping_rows)

with mapping_file.open(encoding="utf-8-sig", newline="") as file:
    saved_rows = list(csv.DictReader(file))

saved = next(row for row in saved_rows if row["source_code"] == "J189")
assert saved["target_concept_id"] == "255848"
assert len(saved_rows) == 72

print("J189 → 255848 Pneumonia")
print("Athena evidence recorded.")
print("Local source vocabulary still requires confirmation.")
print("Saved to:", mapping_file.resolve())

In [ ]:
from datetime import date

item = mapping_inventory["I214"]

item.update({
    "target_concept_id": 4270024,
    "target_concept_name":
        "Acute non-ST segment elevation myocardial infarction",
    "target_domain": "Condition",
    "mapping_status":
        "Athena mapping verified; local source vocabulary pending",
    "confirmed_source_vocabulary": "",
    "evidence_source_vocabulary": "ICD10",
    "evidence_source_concept_id": 45572081,
    "target_vocabulary": "SNOMED",
    "target_standard_status": "Standard",
    "target_validity": "Valid",
    "source_evidence_url":
        "https://athena.ohdsi.org/search-terms/terms/45572081",
    "target_evidence_url":
        "https://athena.ohdsi.org/search-terms/terms/4270024",
    "checked_date": date.today().isoformat(),
})

mapping_rows = [
    mapping_inventory[code]
    for code in sorted(mapping_inventory)
]

with mapping_file.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(
        file, fieldnames=list(mapping_rows[0].keys())
    )
    writer.writeheader()
    writer.writerows(mapping_rows)

with mapping_file.open(encoding="utf-8-sig", newline="") as file:
    saved_rows = list(csv.DictReader(file))

saved_lookup = {row["source_code"]: row for row in saved_rows}

assert len(saved_rows) == 72
assert saved_lookup["J189"]["target_concept_id"] == "255848"
assert saved_lookup["I214"]["target_concept_id"] == "4270024"

print("I214 → 4270024: Athena evidence recorded.")
print("J189 mapping preserved.")
print("Total source codes:", len(saved_rows))
print("Saved to:", mapping_file.resolve())

In [ ]:
import json
from urllib.request import Request, urlopen

vocabulary_base = (
    "https://atlas-demo.ohdsi.org/"
    "WebAPI/vocabulary/SYNPUF1K/"
)

vocabulary_connection_ok = False

try:
    request = Request(
        vocabulary_base + "info",
        headers={"Accept": "application/json"}
    )

    with urlopen(request, timeout=20) as response:
        vocabulary_info = json.load(response)

    print("Connection successful.")
    print("Vocabulary version:", vocabulary_info.get("version", "Unknown"))
    vocabulary_connection_ok = True

except Exception as error:
    print("Connection failed.")
    print(type(error).__name__ + ":", error)

In [ ]:
import runpy

batch_program = runpy.run_path(
    "/Users/shelly/.codex/.chatgpt-projects/"
    "g-p-6a9014bfacb8819198ff0c6cbad00e08/"
    "person2_work/batch_mapping.py",
    run_name="person2_batch"
)

batch_result_folder = batch_program["run_batch"](mapping_file)

In [ ]:
from collections import defaultdict, Counter

candidate_file = batch_result_folder / "batch_mapping_candidates.csv"

with candidate_file.open(encoding="utf-8-sig", newline="") as file:
    batch_candidates = list(csv.DictReader(file))

candidates_by_code = defaultdict(list)

for row in batch_candidates:
    candidates_by_code[row["source_code"]].append(row)

assert len(candidates_by_code) == 72

special_codes = set(
    "U739 W010 X59 Y408 Y409 Y442 Y531 Y9209 Y9222 Y929 "
    "Z312 Z599 Z720 Z8643 Z921 Z952 Z954 M85503".split()
)

review_summary = []

for code, rows in sorted(candidates_by_code.items()):
    reasons = []

    valid_targets = [
        row for row in rows
        if row["target_concept_id"] not in ("", "0")
        and row["target_standard_status"] == "Standard"
        and row["target_validity"] == "Valid"
    ]

    target_ids = {
        row["target_concept_id"] for row in valid_targets
    }

    source_vocabularies = {
        row["source_vocabulary"]
        for row in rows
        if row["source_vocabulary"]
    }

    domains = {
        row["target_domain"] for row in valid_targets
    }

    if not valid_targets:
        reasons.append("No valid standard target")

    if len(target_ids) > 1:
        reasons.append("Multiple distinct target candidates")

    if source_vocabularies == {"ICD10CM"}:
        reasons.append("ICD10CM-only source candidate")

    if any(
        row["source_validity"]
        and row["source_validity"] != "Valid"
        for row in rows
    ):
        reasons.append("Invalid source concept candidate")

    if domains - {"Condition"}:
        reasons.append("Target outside Condition domain")

    if code in special_codes:
        reasons.append("Context, history, service or morphology review")

    if code == "S7211":
        reasons.append("Known fracture meaning conflict across vocabularies")

    review_summary.append({
        "source_code": code,
        "candidate_target_ids": ";".join(sorted(target_ids)),
        "candidate_domains": ";".join(sorted(domains)),
        "candidate_source_vocabularies":
            ";".join(sorted(source_vocabularies)),
        "review_category":
            "Focused review" if reasons else "Candidate available",
        "review_reasons": "; ".join(reasons),
        "local_source_system_status": "Pending confirmation",
    })

review_file = batch_result_folder / "mapping_review_summary.csv"

with review_file.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(
        file, fieldnames=list(review_summary[0].keys())
    )
    writer.writeheader()
    writer.writerows(review_summary)

counts = Counter(
    row["review_category"] for row in review_summary
)

print("Total source codes:", len(review_summary))
print("Candidate available:", counts["Candidate available"])
print("Focused review:", counts["Focused review"])
print("Saved to:", review_file.resolve())

In [ ]:
from collections import Counter

reason_counts = Counter()

for row in review_summary:
    if row["review_category"] == "Focused review":
        for reason in row["review_reasons"].split("; "):
            if reason:
                reason_counts[reason] += 1

print("REVIEW REASONS")
for reason, count in reason_counts.most_common():
    print(f"{reason}: {count}")

print("\nFOCUSED REVIEW CODES")
for row in review_summary:
    if row["review_category"] == "Focused review":
        print(
            row["source_code"],
            "|",
            row["review_reasons"]
        )

In [ ]:
multiple_target_codes = [
    "I071", "I341", "I350", "I48", "R31", "S7211"
]

for code in multiple_target_codes:
    print(f"\n--- {code} ---")

    for row in candidates_by_code[code]:
        print(
            "Source:",
            row["source_vocabulary"],
            "|",
            row["source_concept_name"],
            "| Source validity:",
            row["source_validity"],
        )
        print(
            "Target:",
            row["target_concept_id"],
            "|",
            row["target_concept_name"],
            "| Domain:",
            row["target_domain"],
        )

In [ ]:
original_ids = {
    str(record[field])
    for record in unique_records
    for field in ("Patient_No", "Case_No")
    if record[field] is not None
}

private_columns = {
    "patient_no",
    "case_no",
    "date_offset_days",
    "group_key",
    "group_key.hex",
}

privacy_findings = []
checked_files = list(output_folder.rglob("*.csv"))

for path in checked_files:
    with path.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)

        for column in reader.fieldnames or []:
            if column.strip().lower() in private_columns:
                privacy_findings.append(
                    (path.name, "Private column present")
                )

        for row_number, row in enumerate(reader, start=2):
            if any(
                original_id in str(value)
                for value in row.values()
                if value is not None
                for original_id in original_ids
            ):
                privacy_findings.append(
                    (path.name, f"Original identifier at row {row_number}")
                )

print("CSV files checked:", len(checked_files))
print("Potential identifier leaks:", len(privacy_findings))

for filename, issue in privacy_findings:
    print(filename, "|", issue)

if not privacy_findings:
    print("Original identifier check passed.")

In [ ]:
from datetime import datetime, timedelta

cleaned_file = output_folder / "cleaned_case_visits.csv"

with cleaned_file.open(encoding="utf-8-sig", newline="") as file:
    saved_cleaned = list(csv.DictReader(file))

saved_by_visit = {
    row["visit_occurrence_id"]: row
    for row in saved_cleaned
}

assert len(saved_cleaned) == len(saved_by_visit) == 21

date_checks = 0
date_issues = []

for original in unique_records:
    patient = str(original["Patient_No"])
    case = str(original["Case_No"])

    visit_id = str(
        visit_lookup[(patient, case)]["visit_occurrence_id"]
    )
    saved = saved_by_visit[visit_id]

    offset = timedelta(
        days=int(person_lookup[patient]["date_offset_days"])
    )

    for field in [
        "Adm_DateTime",
        "Dis_DateTime",
        "Dis_DateTime_Planned",
    ]:
        date_checks += 1
        original_value = original[field]
        saved_value = saved[field]

        if original_value is None:
            correct = saved_value == ""
        else:
            expected = original_value + offset

            try:
                actual = datetime.fromisoformat(saved_value)
                correct = actual == expected
            except (ValueError, TypeError):
                correct = False

        if not correct:
            date_issues.append((visit_id, field))

print("Visits checked:", len(saved_cleaned))
print("Date fields checked:", date_checks)
print("Date shift issues:", len(date_issues))

if not date_issues:
    print("All saved dates match the shared patient-specific offsets.")
    print("Missing dates remain missing.")
else:
    print("Date checks failed. Do not share the outputs yet.")

In [ ]:
import shutil
import json
from datetime import datetime

assert not privacy_findings, "Resolve identifier findings first."
assert not date_issues, "Resolve date-shift issues first."

handoff_folder = raw_folder / (
    "Group_Review_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
handoff_folder.mkdir()

files_to_include = [
    "cleaned_case_visits.csv",
    "visit_occurrence_provisional.csv",
    "diagnosis_events.csv",
    "diagnosis_mapping_worklist.csv",
]

# Confirm all required files exist before copying.
for name in files_to_include:
    assert (output_folder / name).is_file(), f"Missing file: {name}"

for name in files_to_include:
    shutil.copy2(output_folder / name, handoff_folder / name)

report = {
    "release_scope": "Private group review only",
    "privacy_status": "Pseudonymized; not approved for public release",
    "original_identifier_findings": len(privacy_findings),
    "date_fields_checked": date_checks,
    "date_shift_issues": len(date_issues),
    "source_visit_records": 21,
    "mapping_status": "Provisional; source coding system unspecified",
    "included_files": files_to_include,
}

(handoff_folder / "privacy_check_report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

(handoff_folder / "README_EN.txt").write_text(
    "PERSON 2 - PRIVATE GROUP REVIEW\n\n"
    "Patient and visit IDs use the shared registries.\n"
    "Dates use the shared patient-specific offsets.\n"
    "Original identifiers and private registry files are excluded.\n"
    "These clinical data remain pseudonymized and are not approved "
    "for public release.\n\n"
    "Visit classification and diagnosis mappings remain provisional.\n"
    "Diagnosis events are source events, not a completed "
    "CONDITION_OCCURRENCE table.\n",
    encoding="utf-8"
)

print("Group review folder created:")
print(handoff_folder.resolve())

for path in sorted(handoff_folder.iterdir()):
    print(path.name)

In [ ]:
code_disposition = []

for row in review_summary:
    code = row["source_code"]

    if code in special_codes:
        action = "Routing review"
        note = (
            "Review context, history, status, service or morphology. "
            "Do not automatically load as an active condition."
        )
    elif row["review_category"] == "Focused review":
        action = "Unresolved condition mapping"
        note = row["review_reasons"]
    else:
        action = "Condition mapping candidate"
        note = (
            "Candidate available; local source coding system "
            "has not been confirmed."
        )

    code_disposition.append({
        "source_code": code,
        "event_count": mapping_inventory[code]["event_count"],
        "processing_action": action,
        "notes": note,
        "final_mapping_approved": "No",
    })

assert len(code_disposition) == 72
assert sum(row["event_count"] for row in code_disposition) == 147

disposition_file = output_folder / "diagnosis_code_disposition.csv"

with disposition_file.open(
    "w", encoding="utf-8-sig", newline=""
) as file:
    writer = csv.DictWriter(
        file, fieldnames=list(code_disposition[0].keys())
    )
    writer.writeheader()
    writer.writerows(code_disposition)

shutil.copy2(
    disposition_file,
    handoff_folder / disposition_file.name
)

counts = Counter(
    row["processing_action"] for row in code_disposition
)

for action, count in counts.items():
    print(f"{action}: {count}")

print("Diagnosis events accounted for:", sum(
    row["event_count"] for row in code_disposition
))
print("Saved to:", disposition_file.resolve())

In [ ]:
condition_occurrence = []
condition_review_notes = []
routing_events = []

disposition_lookup = {
    row["source_code"]: row["processing_action"]
    for row in code_disposition
}

for event in diagnosis_events:
    code = event["source_code"]
    action = disposition_lookup[code]

    if action == "Routing review":
        routing_events.append(dict(event))
        continue

    target_id = 0

    if action == "Condition mapping candidate":
        valid_ids = {
            int(row["target_concept_id"])
            for row in candidates_by_code[code]
            if row["target_concept_id"] not in ("", "0")
            and row["target_standard_status"] == "Standard"
            and row["target_validity"] == "Valid"
            and row["target_domain"] == "Condition"
        }

        assert len(valid_ids) == 1, (
            f"{code}: expected exactly one Condition candidate."
        )
        target_id = next(iter(valid_ids))

    # Reuse the stable source-event number for this provisional table.
    condition_id = event["diagnosis_event_id"]

    condition_occurrence.append({
        "condition_occurrence_id": condition_id,
        "person_id": event["person_id"],
        "condition_concept_id": target_id,
        "condition_start_date": event["date_proxy"],
        "condition_start_datetime": None,
        "condition_end_date": None,
        "condition_end_datetime": None,
        "condition_type_concept_id": 32817,
        "condition_status_concept_id":
            32902 if event["diagnosis_role"] == "Primary" else 32908,
        "stop_reason": None,
        "provider_id": None,
        "visit_occurrence_id": event["visit_occurrence_id"],
        "visit_detail_id": None,
        "condition_source_value": code,
        "condition_source_concept_id": 0,
        "condition_status_source_value": event["diagnosis_role"],
    })

    condition_review_notes.append({
        "condition_occurrence_id": condition_id,
        "source_code": code,
        "review_status": action,
        "final_mapping_approved": "No",
        "source_vocabulary": "Not specified in supplied dataset",
        "date_basis": "Shifted admission date proxy",
        "provenance_assumption": "32817 EHR; requires confirmation",
    })

assert len(condition_occurrence) + len(routing_events) == 147
assert len({
    row["condition_occurrence_id"]
    for row in condition_occurrence
}) == len(condition_occurrence)

visit_person_lookup = {
    row["visit_occurrence_id"]: row["person_id"]
    for row in visit_occurrence
}

for row in condition_occurrence:
    assert visit_person_lookup[row["visit_occurrence_id"]] == row["person_id"]

def save_review_csv(name, records):
    path = output_folder / name

    with path.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(
            file, fieldnames=list(records[0].keys())
        )
        writer.writeheader()
        writer.writerows(records)

    with path.open(encoding="utf-8-sig", newline="") as file:
        assert len(list(csv.DictReader(file))) == len(records)

    return path

condition_file = save_review_csv(
    "condition_occurrence_provisional.csv",
    condition_occurrence
)
save_review_csv(
    "condition_review_notes.csv",
    condition_review_notes
)
save_review_csv(
    "routing_events_review.csv",
    routing_events
)

print("Provisional Condition rows:", len(condition_occurrence))
print("Rows with candidate concepts:", sum(
    row["condition_concept_id"] != 0 for row in condition_occurrence
))
print("Rows with concept ID 0:", sum(
    row["condition_concept_id"] == 0 for row in condition_occurrence
))
print("Routing review events:", len(routing_events))
print("Total events accounted for:",
      len(condition_occurrence) + len(routing_events))
print("Saved to:", condition_file.resolve())

In [ ]:
new_files = [
    "condition_occurrence_provisional.csv",
    "condition_review_notes.csv",
    "routing_events_review.csv",
]

# Check saved files for original identifiers and private columns.
for name in new_files:
    path = output_folder / name

    with path.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)

        assert not (
            {column.lower() for column in reader.fieldnames}
            & private_columns
        ), f"Private column found in {name}"

        for row in reader:
            assert not any(
                original_id in str(value)
                for value in row.values()
                if value is not None
                for original_id in original_ids
            ), f"Original identifier found in {name}"

# Check saved Condition records against the saved Visit table.
with visit_file.open(encoding="utf-8-sig", newline="") as file:
    saved_visit_lookup = {
        row["visit_occurrence_id"]: row
        for row in csv.DictReader(file)
    }

with condition_file.open(encoding="utf-8-sig", newline="") as file:
    saved_conditions = list(csv.DictReader(file))

assert len(saved_conditions) == 92

for row in saved_conditions:
    visit = saved_visit_lookup[row["visit_occurrence_id"]]

    assert row["person_id"] == visit["person_id"]
    assert row["condition_start_date"] == visit["visit_start_date"]
    assert row["condition_start_datetime"] == ""
    assert row["condition_end_date"] == ""
    assert row["condition_end_datetime"] == ""

# Copy checked files into the existing group review folder.
for name in new_files:
    shutil.copy2(
        output_folder / name,
        handoff_folder / name
    )

report.update({
    "provisional_condition_rows": 92,
    "condition_rows_with_candidate_concepts": 68,
    "condition_rows_with_concept_zero": 24,
    "routing_review_events": 55,
    "diagnosis_events_accounted_for": 147,
    "new_files_original_identifier_check": "Passed",
    "saved_condition_visit_linkage_check": "Passed",
    "saved_condition_date_proxy_check": "Passed",
})

report["included_files"] = sorted(
    path.name for path in handoff_folder.glob("*.csv")
)

(handoff_folder / "privacy_check_report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

(handoff_folder / "README_EN.txt").write_text(
    "PERSON 2 - PRIVATE GROUP REVIEW\n\n"
    "Shared patient/visit IDs and patient-specific date offsets "
    "have been applied.\n"
    "Original identifiers and private registries are excluded "
    "from this review package.\n"
    "The data remain pseudonymized; public-release anonymization "
    "has not been established.\n\n"
    "There are 21 provisional visits and 147 diagnosis source events.\n"
    "The provisional Condition table contains 92 rows: "
    "68 with candidate concepts and 24 with concept ID 0.\n"
    "Another 55 events require routing review.\n"
    "All diagnosis source events are accounted for.\n\n"
    "The source coding system is not specified in the supplied data.\n"
    "Mappings and EHR provenance remain provisional.\n"
    "Condition start dates use shifted admission-date proxies; "
    "disease resolution dates are not inferred.\n",
    encoding="utf-8"
)

print("New files: original identifier check passed.")
print("Saved Condition linkage and date checks passed.")
print("Group review package updated:", handoff_folder.resolve())
print("CSV files included:", len(report["included_files"]))